In [1]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
from google.cloud import bigquery

In [2]:
# =============================================================
# CẤU HÌNH: gold_zone
# =============================================================
PROJECT_ID = "project-7f16ff1d-9026-4799-bfa"
BUCKET = "amazon-reviews-lakehouse-data"
DATASET_ID = "gold_zone"

REVIEW_DATA_PATH = f"gs://{BUCKET}/silver-zone/review-data"
# META_DATA_PATH = f"gs://{BUCKET}/silver-zone/meta-data"

CURRENT_TS = 1704067200

# Khởi tạo Spark
spark = DataprocSparkSession.builder.getOrCreate()

█████████████████████▎                                                          

In [4]:
# Khởi tạo BigQuery Client để kiểm tra dataset
bq_client = bigquery.Client(project=PROJECT_ID)

def check_and_create_dataset():
    dataset_full_id = f"{PROJECT_ID}.{DATASET_ID}"
    try:
        bq_client.get_dataset(dataset_full_id)
        print(f"✅ Dataset '{DATASET_ID}' đã sẵn sàng.")
    except Exception:
        print(f"⚠️ Không tìm thấy '{DATASET_ID}'. Đang tự động tạo mới...")
        dataset = bigquery.Dataset(dataset_full_id)
        dataset.location = "asia-southeast1" # Đảm bảo cùng vùng với GCS
        bq_client.create_dataset(dataset, timeout=30)
        print(f"✅ Đã tạo thành công Dataset: {DATASET_ID}")

check_and_create_dataset()

✅ Dataset 'gold_zone' đã sẵn sàng.


In [5]:
def build_gold_user_sentiment_features():
    # Đọc dữ liệu
    df = spark.read.option("recursiveFileLookup", "true").parquet(REVIEW_DATA_PATH)
    print(f"Đã load {df.count():,} bản ghi review")

    # ====================== DÙNG RATING LÀM SENTIMENT SCORE ======================
    # Chuẩn hóa rating từ 1-5 thành khoảng -1 đến +1
    df = df.withColumn("sentiment_score",
                       (F.col("rating") - 3.0) / 2.0)   # 1→-1, 3→0, 5→+1

    print("Đã dùng rating để tính sentiment_score")

    # Tính đúng 5 chỉ số
    print("Tính avg_sentiment_score, sentiment_intensity_avg, positive_ratio, negative_ratio...")
    user_features = df.groupBy("user_id").agg(
        F.avg("sentiment_score").alias("avg_sentiment_score"),
        F.avg(F.abs("sentiment_score")).alias("sentiment_intensity_avg"),
        (F.sum(F.when(F.col("sentiment_score") > 0.2, 1).otherwise(0)) / F.count("*")).alias("positive_ratio"),
        (F.sum(F.when(F.col("sentiment_score") < -0.2, 1).otherwise(0)) / F.count("*")).alias("negative_ratio")
    )

    # emotion_trend_slope (30 ngày gần nhất)
    print("Tính emotion_trend_slope...")
    THIRTY_DAYS_AGO = CURRENT_TS - (30 * 86400)
    df_recent = df.filter(F.col("timestamp").cast("long") >= THIRTY_DAYS_AGO)
    df_recent = df_recent.withColumn("time", F.col("timestamp").cast("long") / 86400.0)

    trend = df_recent.groupBy("user_id").agg(
        F.when(
            F.var_pop("time") != 0,
            F.covar_pop("time", "sentiment_score") / F.var_pop("time")
        ).otherwise(0.0).alias("emotion_trend_slope")
    )

    final_bt2 = user_features.join(trend, "user_id", "left")

    # Ghi vào BigQuery
    print(f"Đang ghi bảng: {PROJECT_ID}.{DATASET_ID}.user_sentiment_features")
    final_bt2.write.format("bigquery") \
        .option("table", f"{PROJECT_ID}.{DATASET_ID}.user_sentiment_features") \
        .option("temporaryGcsBucket", BUCKET) \
        .mode("overwrite") \
        .save()

    print("HOÀN TẤT!")
    final_bt2.show(5, truncate=False)

# === CHẠY ===
build_gold_user_sentiment_features()

  0%|           0/1 Tasks

Đã load 535,264,768 bản ghi review
Đã dùng rating để tính sentiment_score
Tính avg_sentiment_score, sentiment_intensity_avg, positive_ratio, negative_ratio...
Tính emotion_trend_slope...
Đang ghi bảng: project-7f16ff1d-9026-4799-bfa.gold_zone.user_sentiment_features


  0%|           0/1558 Tasks

HOÀN TẤT!


  0%|           0/1558 Tasks

+----------------------------+-------------------+-----------------------+------------------+--------------------+-------------------+
|user_id                     |avg_sentiment_score|sentiment_intensity_avg|positive_ratio    |negative_ratio      |emotion_trend_slope|
+----------------------------+-------------------+-----------------------+------------------+--------------------+-------------------+
|AE22BKXGDZLDVUMLJMY5PF3OJW7Q|0.6904761904761905 |0.7857142857142857     |0.8095238095238095|0.047619047619047616|NULL               |
|AE26NU3NH4WQBW4J7QYBGJRMUGEA|0.2222222222222222 |0.6666666666666666     |0.5555555555555556|0.3333333333333333  |NULL               |
|AE2EBRKEHCWAYBPIKVY44UXYAKSA|0.7                |0.7                    |0.8               |0.0                 |NULL               |
|AE2IDNPQQHENA4BN3HV6F5Y2GEZA|0.3979591836734694 |0.8809523809523809     |0.6462585034013606|0.272108843537415   |NULL               |
|AE2U6LM5QEIA65XJZ3DGHJGAC6ZQ|0.8111111111111111 |0.888